In [ ]:
!pip install deepeval openai bert-score

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.3/81.3 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.0/572.0 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.9/55.9 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.9/118.9 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.2/65.2 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

In [ ]:
!git clone https://github.com/adamcochrane/Dissertation.git

Cloning into 'Dissertation'...
remote: Enumerating objects: 83, done.
remote: Counting objects: 100% (83/83), done.
remote: Compressing objects: 100% (80/80), done.
remote: Total 83 (delta 42), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (83/83), 6.87 MiB | 1.84 MiB/s, done.
Resolving deltas: 100% (42/42), done.


In [ ]:
import pandas as pd
from google.colab import files, userdata
from openai import OpenAI
import os
from deepeval.test_case import LLMTestCase
from deepeval.metrics import FaithfulnessMetric
from bert_score import score
import ast
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

os.environ["OPENAI_API_KEY"] = userdata.get('OpenAI_key')

file_name = f"/content/Dissertation/csv files/prompting-llama-baseline.csv"
test_responses = pd.read_csv(file_name)


faithfulness_results = {}
faithfulness_reasons = {}
bertP_results = {}
bleu_results = {}
short_answers_results = {}


# FAITHFULNESS METRIC
faithful_metric = FaithfulnessMetric(
  threshold=0.7,
  model="gpt-4o-mini",
  include_reason=True
)
def getFaithfulness(prompt_id, question, response, context):
  test_case = LLMTestCase(
    input = question,
    actual_output = response,
    retrieval_context = [context]
  )
  faithful_metric.measure(test_case)
  faithfulness_results[prompt_id] = faithful_metric.score
  if faithful_metric.score < 1.0:
    faithfulness_reasons[prompt_id] = faithful_metric.reason
  else:
    faithfulness_reasons[prompt_id] = ""



#BertMetric
def getBertScore(prompt_id, response, context):
  answers = [response]
  expected_answers = [context]
  P, R, F1 = score(answers, expected_answers, lang='en', rescale_with_baseline=True)
  bertP_results[prompt_id] = P.mean().item()



#Bleu metric
def getBleuScore(prompt_id, response, answers):
  reference = answers.split()
  candidate = response.split()
  smoothing = SmoothingFunction().method1
  average_bleu_score = sentence_bleu(reference, candidate, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function = smoothing)
  bleu_results[prompt_id] = average_bleu_score



#Personal metric
def getAnswersScore(prompt_id, response, answers):
  response_lower = response.lower()
  answers_present = set()
  unique_answers = set([answer.lower() for answer in answers])

  for answer in unique_answers:
    if (answer in response_lower):
      answers_present.add(answer)

  score = len(answers_present) / len(unique_answers)
  # print("Answers present score = ", score)
  short_answers_results[prompt_id] = score




for id in test_responses["prompt_id"]:
  if pd.notna(id):
    row_index = test_responses[test_responses["prompt_id"] == id].index[0]
    print("Row index = ", row_index)
    question = test_responses.iloc[row_index]["question"]
    response = test_responses.iloc[row_index]["response"]
    context = test_responses.iloc[row_index]["retrieval_context"]
    short_answers = ast.literal_eval(test_responses.iloc[row_index]["dataset_answers"])

    getFaithfulness(id, question, response, context)
    getBertScore(id, response, [context])
    getBleuScore(id, response, context)
    getAnswersScore(id, response, short_answers)


test_responses["bert_score"] = test_responses["prompt_id"].map(bertP_results)
test_responses["bleu_score"] = test_responses["prompt_id"].map(bleu_results)
test_responses["short_answers_score"] = test_responses["prompt_id"].map(short_answers_results)
test_responses["faithfulness_score"] = test_responses["prompt_id"].map(faithfulness_results)
test_responses["faithfulness_reason"] = test_responses["prompt_id"].map(faithfulness_reasons)

file_name = f"/content/Dissertation/baseline-faithful-metrics.csv"
test_responses.to_csv(file_name, index=False)
files.download(file_name)







Output()

Row index =  0


Faithfulness score =  1.0


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

...

Row index =  1087


Faithfulness score =  1.0


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>